In [1]:
# Input Raw Data On Order Report
import pandas as pd
OnOrderReport_df = pd.read_excel("Raw Data.xlsx", skiprows = 3) 
print(OnOrderReport_df.columns)

Index(['Customer Name', 'Customer Abbreviation', 'Customer Po',
       'Customer Po Line', 'Date Entered', 'Order No', 'Due Date Code',
       'Part No', 'Description', 'Revised Due Date', 'ESD', 'Qty Open',
       'Unit Price', 'Total Value Open', 'Qty Resvered', 'Qty Picked',
       'Qty Backordered', 'Order date', 'Current Sales Status Code',
       'Fin Business Code', 'Carrier Code'],
      dtype='object')


In [2]:
# Make new Sheet PTC Normal
# --- Filter by Customer Name ---
PTCNormal_df = OnOrderReport_df[
    OnOrderReport_df["Customer Name"] == "PT. CIPTA ANDALAN TEKNINDO"
]

In [3]:
# List of new columns to add
new_columns = [
    "Full/Partial",
    "Mon Reqrd.",
    "Mon.Reqrd.2 weeks ago",
    "chk",
    "Year Reqrd.",
    "Mon ListPO",
    "Year 2 weeks ago",
    "chk. Year"
]

# Find position of 'Qty Backordered'
insert_pos = PTCNormal_df.columns.get_loc("Qty Backordered") + 1

# Insert columns one by one (keeps order)
for i, col in enumerate(new_columns):
    PTCNormal_df.insert(insert_pos + i, col, "")
    


In [4]:
# Find position of 'Part No'
insert_pos = PTCNormal_df.columns.get_loc("Part No") + 1

# 1. PN used (same as Part No)
PTCNormal_df.insert(
    insert_pos,
    "PN used",
    PTCNormal_df["Part No"]
)

# 2. Order.PN (merge Order No and Part No)
PTCNormal_df.insert(
    insert_pos + 1,
    "Order.PN",
    PTCNormal_df["Order No"].astype(str) + "." + PTCNormal_df["Part No"].astype(str)
)

# ---- DO NOT create PN AL78 here ----


# Read AL78 master file
al78_df = pd.read_excel("LongPN to AL78 PN.xlsx")

# Clean join keys
PTCNormal_df["PN used"] = (
    PTCNormal_df["PN used"]
    .astype(str)
    .str.strip()
)

al78_df["PART_NO"] = (
    al78_df["PART_NO"]
    .astype(str)
    .str.strip()
)

# Merge PN AL78
PTCNormal_df = PTCNormal_df.merge(
    al78_df[["PART_NO", "PN AL78"]],
    left_on="PN used",
    right_on="PART_NO",
    how="left"
)

# Drop helper column
PTCNormal_df.drop(columns=["PART_NO"], inplace=True)

# Reorder PN AL78 to be right after Order.PN
cols = list(PTCNormal_df.columns)

cols.remove("PN AL78")

order_pn_index = cols.index("Order.PN")

cols.insert(order_pn_index + 1, "PN AL78")

PTCNormal_df = PTCNormal_df[cols]



C:\Users\Brandon\AppData\Local\Temp\ipykernel_11324\1397811444.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  PTCNormal_df["PN used"] = (


In [5]:
#Full/Partial
import numpy as np

PTCNormal_df["Full/Partial"] = np.select(
    [
        PTCNormal_df["Qty Resvered"] == 0,
        PTCNormal_df["Qty Resvered"] == PTCNormal_df["Qty Open"]
    ],
    [
        "00",
        "Full"
    ],
    default="Partial"
)


In [6]:
two_weeks_df = pd.read_excel(
    "2 weeks ago.xlsx",
    sheet_name="PTC Normal",
    skiprows=3
)
PTCNormal_df["Order.PN"] = (
    PTCNormal_df["Order.PN"]
    .astype(str)
    .str.strip()
)

two_weeks_df["Order.PN"] = (
    two_weeks_df["Order.PN"]
    .astype(str)
    .str.strip()
)
two_weeks_lookup = (
    two_weeks_df[["Order.PN", "Mon\nReqrd."]]
    .dropna(subset=["Order.PN"])
    .drop_duplicates(subset=["Order.PN"], keep="last")
)
if "Mon.Reqrd.2 weeks ago" in PTCNormal_df.columns:
    PTCNormal_df.drop(columns=["Mon.Reqrd.2 weeks ago"], inplace=True)
PTCNormal_df = PTCNormal_df.merge(
    two_weeks_lookup,
    on="Order.PN",
    how="left",
    validate="many_to_one"   # 🚨 protects row count
)
# Rename
PTCNormal_df.rename(
    columns={"Mon\nReqrd.": "Mon.Reqrd.2 weeks ago"},
    inplace=True
)

# Move column right after 'Mon Reqrd.'
cols = list(PTCNormal_df.columns)
cols.remove("Mon.Reqrd.2 weeks ago")

idx = cols.index("Mon Reqrd.")
cols.insert(idx + 1, "Mon.Reqrd.2 weeks ago")

PTCNormal_df = PTCNormal_df[cols]


In [7]:
# --- Read PO master file ---
po_df = pd.read_excel("PO PTC to 06Dec2025.xlsx")

# --- Clean join keys ---
PTCNormal_df["Customer Po"] = (
    PTCNormal_df["Customer Po"]
    .astype(str)
    .str.strip()
)

po_df["PO No"] = (
    po_df["PO No"]
    .astype(str)
    .str.strip()
)

# --- Build lookup table (Excel VLOOKUP behavior) ---
po_lookup = (
    po_df[["PO No", "MON"]]
    .dropna(subset=["PO No"])
    .drop_duplicates(subset=["PO No"], keep="last")
)

# --- Remove existing Mon ListPO to avoid duplication ---
if "Mon ListPO" in PTCNormal_df.columns:
    PTCNormal_df.drop(columns=["Mon ListPO"], inplace=True)

# --- Merge ---
PTCNormal_df = PTCNormal_df.merge(
    po_lookup,
    left_on="Customer Po",
    right_on="PO No",
    how="left",
    validate="many_to_one"
)

# --- Rename & cleanup ---
PTCNormal_df.rename(columns={"MON": "Mon ListPO"}, inplace=True)
PTCNormal_df.drop(columns=["PO No"], inplace=True)

# --- Move Mon ListPO to the right of Year Reqrd. ---
cols = list(PTCNormal_df.columns)

cols.remove("Mon ListPO")

year_reqrd_index = cols.index("Year Reqrd.")

cols.insert(year_reqrd_index + 1, "Mon ListPO")

PTCNormal_df = PTCNormal_df[cols]


In [8]:
# --- Read 2 weeks ago file ---
two_weeks_df = pd.read_excel(
    "2 weeks ago.xlsx",
    sheet_name="PTC Normal",
    skiprows=3
)

# --- Clean join keys ---
PTCNormal_df["Order.PN"] = (
    PTCNormal_df["Order.PN"]
    .astype(str)
    .str.strip()
)

two_weeks_df["Order.PN"] = (
    two_weeks_df["Order.PN"]
    .astype(str)
    .str.strip()
)

# --- Prepare lookup table (Excel VLOOKUP behavior) ---
year_lookup = (
    two_weeks_df[["Order.PN", "Year\nReqrd."]]
    .dropna(subset=["Order.PN"])
    .drop_duplicates(subset=["Order.PN"], keep="last")
)

# --- Remove existing column to avoid duplication ---
if "Year 2 weeks ago" in PTCNormal_df.columns:
    PTCNormal_df.drop(columns=["Year 2 weeks ago"], inplace=True)

# --- Merge ---
PTCNormal_df = PTCNormal_df.merge(
    year_lookup,
    on="Order.PN",
    how="left",
    validate="many_to_one"
)

# --- Rename to target column ---
PTCNormal_df.rename(
    columns={"Year\nReqrd.": "Year 2 weeks ago"},
    inplace=True
)


In [9]:
import pandas as pd
import numpy as np
from pandas.tseries.offsets import DateOffset

# Ensure datetime
PTCNormal_df["Order date"] = pd.to_datetime(PTCNormal_df["Order date"], errors="coerce")
order_year = PTCNormal_df["Order date"].dt.year

# Due Date Code rules
due_date_days = {"CD":1,"EC":5,"WA":5,"MT":10,"W5":10,"S1":70,"S3":70,"S4":70,"S5":70,"MO":35}
days_to_add = PTCNormal_df["Due Date Code"].map(due_date_days)

# Formula date (lowest priority)
formula_date = PTCNormal_df["Order date"] + pd.to_timedelta(days_to_add, unit="D")
formula_month = formula_date.dt.month
formula_year  = formula_date.dt.year

# Normalize override sources
mon_2w = pd.to_numeric(PTCNormal_df["Mon.Reqrd.2 weeks ago"].replace("", np.nan), errors="coerce")
mon_listpo = pd.to_numeric(PTCNormal_df["Mon ListPO"].replace("", np.nan), errors="coerce")
year_2w = pd.to_numeric(PTCNormal_df.get("Year 2 weeks ago", pd.Series()).replace("", np.nan), errors="coerce")

# STEP 1: Decide FINAL Month
final_mon = formula_month.copy()
mask_po = mon_2w.isna() & mon_listpo.notna() & (mon_listpo != 0)
final_mon.loc[mask_po] = mon_listpo[mask_po]
mask_2w = mon_2w.notna()
final_mon.loc[mask_2w] = mon_2w[mask_2w]

# STEP 2: Decide FINAL Year
final_year = formula_year.copy()
final_year.loc[mask_po] = order_year[mask_po]
final_year.loc[year_2w.notna()] = year_2w[year_2w.notna()]

# STEP 3: Build FINAL DATE
req_date = pd.to_datetime(dict(year=final_year, month=final_mon, day=1), errors="coerce")

# STEP 4: Apply 7-month cap
min_allowed_date = pd.Timestamp.today().normalize() - DateOffset(months=7)
req_date = req_date.where(req_date >= min_allowed_date, min_allowed_date)

# STEP 5: Assign outputs
PTCNormal_df["Mon Reqrd."] = req_date.dt.month
PTCNormal_df["Year Reqrd."] = req_date.dt.year

# STEP 6: Clean dtype (NO .0)
PTCNormal_df["Mon Reqrd."] = pd.to_numeric(PTCNormal_df["Mon Reqrd."], errors="coerce").astype("Int64").astype(object)
PTCNormal_df["Year Reqrd."] = pd.to_numeric(PTCNormal_df["Year Reqrd."], errors="coerce").astype("Int64").astype(object)


In [10]:
# import pandas as pd
# import numpy as np
# from pandas.tseries.offsets import DateOffset

# # -------------------------------------------------
# # Ensure datetime
# # -------------------------------------------------
# PTCNormal_df["Order date"] = pd.to_datetime(
#     PTCNormal_df["Order date"], errors="coerce"
# )

# order_year = PTCNormal_df["Order date"].dt.year

# # -------------------------------------------------
# # Due Date Code rules
# # -------------------------------------------------
# due_date_days = {
#     "CD": 1, "EC": 5, "WA": 5,
#     "MT": 10, "W5": 10,
#     "S1": 70, "S3": 70, "S4": 70, "S5": 70,
#     "MO": 35
# }

# days_to_add = PTCNormal_df["Due Date Code"].map(due_date_days)

# # -------------------------------------------------
# # Formula date (lowest priority)
# # -------------------------------------------------
# formula_date = (
#     PTCNormal_df["Order date"]
#     + pd.to_timedelta(days_to_add, unit="D")
# )

# formula_month = formula_date.dt.month
# formula_year  = formula_date.dt.year

# # -------------------------------------------------
# # Normalize override sources
# # -------------------------------------------------
# mon_2w = pd.to_numeric(
#     PTCNormal_df["Mon.Reqrd.2 weeks ago"], errors="coerce"
# )

# mon_listpo = pd.to_numeric(
#     PTCNormal_df["Mon ListPO"], errors="coerce"
# )

# year_2w = pd.to_numeric(
#     PTCNormal_df.get("Year 2 weeks ago"), errors="coerce"
# )

# # -------------------------------------------------
# # STEP 1: Decide FINAL Month (PRIORITY)
# # -------------------------------------------------
# final_mon = formula_month.copy()

# # Priority 2: Mon ListPO
# mask_po = mon_2w.isna() & mon_listpo.notna() & (mon_listpo != 0)
# final_mon.loc[mask_po] = mon_listpo[mask_po]

# # Priority 1: 2 weeks ago
# mask_2w = mon_2w.notna()
# final_mon.loc[mask_2w] = mon_2w[mask_2w]

# # -------------------------------------------------
# # STEP 2: Decide FINAL Year
# # -------------------------------------------------
# final_year = formula_year.copy()

# # If month came from PO → year = Order date year
# final_year.loc[mask_po] = order_year[mask_po]

# # If 2 weeks ago year exists → use it
# final_year.loc[year_2w.notna()] = year_2w[year_2w.notna()]

# # -------------------------------------------------
# # STEP 3: Build FINAL required DATE (ONCE)
# # -------------------------------------------------
# req_date = pd.to_datetime(
#     dict(
#         year=final_year,
#         month=final_mon,
#         day=1
#     ),
#     errors="coerce"
# )

# # -------------------------------------------------
# # STEP 4: Apply 7-month CAP (LAST)
# # -------------------------------------------------
# min_allowed_date = pd.Timestamp.today().normalize() - DateOffset(months=7)

# req_date = req_date.where(
#     req_date >= min_allowed_date,
#     min_allowed_date
# )

# # -------------------------------------------------
# # STEP 5: Assign outputs
# # -------------------------------------------------
# PTCNormal_df["Mon Reqrd."] = req_date.dt.month
# PTCNormal_df["Year Reqrd."] = req_date.dt.year

# # -------------------------------------------------
# # Invalid Due Date Code
# # -------------------------------------------------
# PTCNormal_df.loc[
#     days_to_add.isna(),
#     ["Mon Reqrd.", "Year Reqrd."]
# ] = "???"

# # -------------------------------------------------
# # Clean dtype (NO .0)
# # -------------------------------------------------
# PTCNormal_df["Mon Reqrd."] = (
#     pd.to_numeric(PTCNormal_df["Mon Reqrd."], errors="coerce")
#     .astype("Int64")
#     .astype(object)
#     .where(days_to_add.notna(), "???")
# )

# PTCNormal_df["Year Reqrd."] = (
#     pd.to_numeric(PTCNormal_df["Year Reqrd."], errors="coerce")
#     .astype("Int64")
#     .astype(object)
#     .where(days_to_add.notna(), "???")
# )


In [11]:
# from pandas.tseries.offsets import DateOffset
# import pandas as pd
# import numpy as np

# # -------------------------------------------------
# # Normalize override source
# # -------------------------------------------------
# year_2w = pd.to_numeric(
#     PTCNormal_df["Year 2 weeks ago"], errors="coerce"
# )

# # -------------------------------------------------
# # Step 1: Build base required DATE from FINAL Mon Reqrd.
# # -------------------------------------------------
# base_req_date = pd.to_datetime(
#     dict(
#         year=PTCNormal_df["Order date"].dt.year,
#         month=pd.to_numeric(PTCNormal_df["Mon Reqrd."], errors="coerce"),
#         day=1
#     ),
#     errors="coerce"
# )

# # -------------------------------------------------
# # Step 2: Apply 7-month minimum CAP (DATE-LEVEL)
# # -------------------------------------------------
# min_allowed_date = pd.Timestamp.today().normalize() - DateOffset(months=7)

# base_req_date = base_req_date.where(
#     base_req_date >= min_allowed_date,
#     min_allowed_date
# )

# # -------------------------------------------------
# # Step 3: Extract year (formula fallback)
# # -------------------------------------------------
# fallback_year = base_req_date.dt.year

# PTCNormal_df["Year Reqrd."] = fallback_year

# # -------------------------------------------------
# # Step 4: Priority override (2 weeks ago)
# # -------------------------------------------------
# mask_2w = year_2w.notna()
# PTCNormal_df.loc[mask_2w, "Year Reqrd."] = year_2w[mask_2w]

# # -------------------------------------------------
# # Step 5: Invalid Due Date Code
# # -------------------------------------------------
# PTCNormal_df.loc[
#     PTCNormal_df["Mon Reqrd."] == "???",
#     "Year Reqrd."
# ] = "???"

# # -------------------------------------------------
# # Step 6: Clean dtype (no .0, Excel-safe)
# # -------------------------------------------------
# PTCNormal_df["Year Reqrd."] = (
#     pd.to_numeric(PTCNormal_df["Year Reqrd."], errors="coerce")
#     .astype("Int64")
#     .astype(object)
#     .where(PTCNormal_df["Mon Reqrd."] != "???", "???")
# )


In [12]:
# from pandas.tseries.offsets import DateOffset
# import pandas as pd
# import numpy as np

# # -------------------------------------------------
# # Ensure datetime
# # -------------------------------------------------
# PTCNormal_df["Order date"] = pd.to_datetime(
#     PTCNormal_df["Order date"], errors="coerce"
# )

# # -------------------------------------------------
# # Due Date Code rules
# # -------------------------------------------------
# due_date_days = {
#     "CD": 1, "EC": 5, "WA": 5,
#     "MT": 10, "W5": 10,
#     "S1": 70, "S3": 70, "S4": 70, "S5": 70,
#     "MO": 35
# }

# # -------------------------------------------------
# # Base calculated date
# # -------------------------------------------------
# days_to_add = PTCNormal_df["Due Date Code"].map(due_date_days)

# calc_date = (
#     PTCNormal_df["Order date"]
#     + pd.to_timedelta(days_to_add, unit="D")
# )

# # -------------------------------------------------
# # 7-month minimum DATE
# # -------------------------------------------------
# min_allowed_date = pd.Timestamp.today().normalize() - DateOffset(months=7)

# calc_date = calc_date.where(calc_date >= min_allowed_date, min_allowed_date)

# fallback_mon = calc_date.dt.month
# fallback_year = calc_date.dt.year


# # -------------------------------------------------
# # Normalize sources
# # -------------------------------------------------
# mon_2w = pd.to_numeric(PTCNormal_df["Mon.Reqrd.2 weeks ago"], errors="coerce")
# mon_listpo = pd.to_numeric(PTCNormal_df["Mon ListPO"], errors="coerce")

# year_2w = pd.to_numeric(PTCNormal_df.get("Year 2 weeks ago"), errors="coerce")


# # -------------------------------------------------
# # Build Mon Reqrd. (PRIORITY)
# # -------------------------------------------------
# PTCNormal_df["Mon Reqrd."] = fallback_mon
# PTCNormal_df["Year Reqrd."] = fallback_year

# # Priority 2: PO List
# mask_po = mon_listpo.notna() & (mon_listpo != 0)
# PTCNormal_df.loc[mask_po, "Mon Reqrd."] = mon_listpo[mask_po]

# # Priority 1: 2 weeks ago
# mask_2w = mon_2w.notna()
# PTCNormal_df.loc[mask_2w, "Mon Reqrd."] = mon_2w[mask_2w]

# if "Year 2 weeks ago" in PTCNormal_df.columns:
#     PTCNormal_df.loc[year_2w.notna(), "Year Reqrd."] = year_2w[year_2w.notna()]


# # -------------------------------------------------
# # APPLY 7-MONTH CAP (CORRECTLY USING DATE)
# # -------------------------------------------------
# req_date = pd.to_datetime(
#     dict(
#         year=PTCNormal_df["Year Reqrd."],
#         month=PTCNormal_df["Mon Reqrd."],
#         day=1
#     ),
#     errors="coerce"
# )

# cap_mask = req_date < min_allowed_date

# PTCNormal_df.loc[cap_mask, "Mon Reqrd."] = min_allowed_date.month
# PTCNormal_df.loc[cap_mask, "Year Reqrd."] = min_allowed_date.year


# # -------------------------------------------------
# # Invalid Due Date Code
# # -------------------------------------------------
# PTCNormal_df.loc[days_to_add.isna(), ["Mon Reqrd.", "Year Reqrd."]] = "???"


# # -------------------------------------------------
# # Clean dtype (NO .0)
# # -------------------------------------------------
# PTCNormal_df["Mon Reqrd."] = (
#     pd.to_numeric(PTCNormal_df["Mon Reqrd."], errors="coerce")
#     .astype("Int64")
#     .astype(object)
# )

# PTCNormal_df["Year Reqrd."] = (
#     pd.to_numeric(PTCNormal_df["Year Reqrd."], errors="coerce")
#     .astype("Int64")
#     .astype(object)
# )


In [13]:
PTCNormal_df["chk"] = np.where(
    (PTCNormal_df["Mon Reqrd."].notna()) &
    (PTCNormal_df["Mon Reqrd."] == PTCNormal_df["Mon.Reqrd.2 weeks ago"]),
    "OK",
    "NG"
)


In [14]:
import numpy as np

PTCNormal_df["chk. Year"] = np.where(
    (PTCNormal_df["Year Reqrd."].isna()) &
    (PTCNormal_df["Year 2 weeks ago"].isna()),
    "",
    np.where(
        PTCNormal_df["Year Reqrd."] == PTCNormal_df["Year 2 weeks ago"],
        "OK",
        "NG"
    )
)


In [15]:
# Create Order date 1 and Order date 2 from the same source
PTCNormal_df["Order date 1"] = PTCNormal_df["Order date"]
PTCNormal_df["Order date 2"] = PTCNormal_df["Order date"]

# Optional: drop the original column
PTCNormal_df.drop(columns=["Order date"], inplace=True)
final_columns = [
    "Customer Name",
    "Customer Abbreviation",
    "Customer Po",
    "Customer Po Line",
    "Order date 1",
    "Order No",
    "Due Date Code",
    "Part No",
    "PN used",
    "Order.PN",
    "PN AL78",
    "Description",
    "Revised Due Date",
    "ESD",
    "Qty Open",
    "Unit Price",
    "Total Value Open",
    "Qty Resvered",
    "Qty Picked",
    "Qty Backordered",
    "Full/Partial",
    "Mon Reqrd.",
    "Mon.Reqrd.2 weeks ago",
    "chk",
    "Year Reqrd.",
    "Mon ListPO",
    "Year 2 weeks ago",
    "chk. Year",
    "Order date 2",
    "Date Entered",
    "Current Sales Status Code",
    "Fin Business Code",
    "Carrier Code"
]

PTCNormal_df = PTCNormal_df[[c for c in final_columns if c in PTCNormal_df.columns]]

def clean_int_column(df, col):
    df[col] = (
        pd.to_numeric(df[col], errors="coerce")
        .astype("Int64")   # pandas nullable integer
    )

# Apply to affected columns
clean_int_column(PTCNormal_df, "Mon.Reqrd.2 weeks ago")
clean_int_column(PTCNormal_df, "Mon ListPO")
clean_int_column(PTCNormal_df, "Year 2 weeks ago")



In [16]:
from datetime import datetime

today_str = datetime.today().strftime("%Y-%m-%d")
output_file = f"Weekly PTA CGL Revised Open Order Report {today_str}.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    PTCNormal_df.to_excel(
        writer,
        sheet_name="PTC Normal",
        index=False
    )

print(f"File saved as: {output_file}")


File saved as: Weekly PTA CGL Revised Open Order Report 2025-12-18.xlsx


In [17]:
# Make new Sheet Pvt Final

In [18]:
# Make new Sheet to Template